In [9]:
import matplotlib.pyplot as plt
import pandas as pd

from services.db import MongoDB_one_zero_four

plt.rcParams["font.family"] = ["Microsoft JhengHei"]
plt.rcParams["axes.unicode_minus"] = False

pd.set_option("display.max_rows", 15)
pd.set_option("display.max_columns", 10)
pd.set_option("display.width", 100)

In [10]:
db = MongoDB_one_zero_four()

condition = {"header.jobName": {"$regex": "python", "$options": "i"}}
projection = {"header": 1, "condition": 1, "_id": 1}
result = db.select_from_bronze(condition, projection)

In [11]:
test = []
for row in result:
    data = row["condition"]["specialty"]
    data.append({"job_id": row["_id"]})
    test.append(data)

In [12]:
"""
轉換成 df 時是以每個 list 作為一個 column, 所以 df 是以 0, 1, ... list index 作為 col name, 
stack 之後因為 series 只有一行, 所以變成row[row] 的格式? 這種格式經過 df.to_list() 會以[row[row], row[row]] 這樣被拉平? 
會像是 [{}, {}, {}]? 有點神奇, 經過這些操作後撫平了一層巢狀結構
"""
"""
處理這種結構: [[dict, dict], [dict, dict], [dict, dict]]
"""
df = pd.DataFrame(test)
series = df.stack()
df_clean = pd.DataFrame(series.to_list())
print(df_clean)

# print("------------")
# series = pd.Series(test)
# print(series)

            code description job_id
0    12001002016      Github    NaN
1    12001003045      Python    NaN
2    12001003086     PyTorch    NaN
3    12001002018         Git    NaN
4    12001001007       Linux    NaN
..           ...         ...    ...
429          NaN         NaN  8eirr
430  12001002018         Git    NaN
431  12001003045      Python    NaN
432  12001006034     Node.js    NaN
433          NaN         NaN  8vqx4

[434 rows x 3 columns]


In [13]:
# get job_name with job_id 作為映射主表
job_name_include: list[str] = ["python", "後端", "engineer", "資料工程"]
job_name_include_regex = r"|".join(job_name_include)

condition = {"header.jobName": {"$regex": job_name_include_regex, "$options": "i"}}

result = db.select_from_bronze(condition)

In [14]:
df = pd.DataFrame(result[10:20])
test = df.copy()

In [15]:
p = test[["_id", "jobDetail"]]

In [16]:
a = pd.json_normalize(p["jobDetail"])[["salary", "salaryMin", "salaryMax", "salaryType"]]
# axis=1, is mean 以 row 為標準合併, index=0 與 index=0 join 以此類推
b = pd.concat([p, a], axis=1)

df = b.drop(columns="jobDetail", axis=1)
df

,_id,salary,salaryMin,salaryMax,salaryType
0,77phe,待遇面議,0,0,10
1,6z1pf,時薪190元,190,190,30
2,6vazb,待遇面議,0,0,10
3,8w0w8,待遇面議,0,0,10
4,7ize1,"月薪90,000~150,000元",90000,150000,50
5,87ua3,待遇面議,0,0,10
6,8n0gu,"年薪1,100,000~1,300,000元",1100000,1300000,60
7,8w1fo,待遇面議,0,0,10
8,8tlno,"年薪1,000,000元以上",1000000,9999999,60
9,8r9w4,待遇面議,0,0,10
